# Contract-aware features and outcomes

**OPTIONS LAB / FEATURE ENGINEERING**

Build explicit feature inputs and match future observations within each option contract. No shifted prices from unrelated contracts enter the target.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "options_lab").is_dir() else Path.cwd().parent
assert (ROOT / "options_lab").is_dir(), "Run from the repository root or notebooks directory."
sys.path.insert(0, str(ROOT))
from options_lab.data import load_snapshots, make_labels, FEATURES
from options_lab.plotting import set_theme, plot_report
set_theme()
INPUTS = sorted([*ROOT.glob("data/raw/*.csv"), *ROOT.glob("data/snapshots/*.csv")])


## Construct labels

Within each contract, match the next strictly later UTC collection date, at most seven calendar days away. The final observation remains unlabeled. Same-day duplicates cannot become future outcomes.

In [ ]:
snapshots, audit = load_snapshots(INPUTS)
labeled = make_labels(snapshots)
print(f"{labeled.target.notna().sum()} labeled / {len(labeled)} usable observations")
print("Input features:", FEATURES)
display(labeled[["contractSymbol", "date_collected", "exit_date_collected", "future_return", "target"]].head(12))

## Check the evaluation boundary

In [ ]:
from options_lab.data import chronological_split
try:
    train, validation, test = chronological_split(labeled)
    print({"train": len(train), "validation": len(validation), "test": len(test)})
except ValueError as error:
    print("Not ready for evaluation:", error)

## Interpretation

Labels measure the next **observed midpoint change for the same option contract** within seven calendar days. They do not measure expiration ITM. Missing future quotes remain unlabeled. Every experiment uses whole-date train/validation/test splits and purges outcomes that reach the next split. Preprocessing fits on training data only. The majority baseline competes with learned models using validation Brier loss; test data is held out.

The included snapshots have no repeat observations for the same contract. An `insufficient_data` report is the expected result until more history is collected. Synthetic fixtures validate the code in tests; their scores are never published as market evidence.
